In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import traceback

print("=== Using cleaned_images to train ResNet18 model ===")
print("Current working directory:", os.getcwd())

if __name__ == '__main__':
    try:
        # Data directory - using cleaned_images
        DATA_DIR = 'dataset_splits/cleaned_images'
        print(f"Data directory: {DATA_DIR}")
        print(f"Data directory exists: {os.path.exists(DATA_DIR)}")

        # Check if data directory exists
        if not os.path.exists(DATA_DIR):
            print(f"Error: Data directory {DATA_DIR} does not exist")
            exit(1)

        # Check subdirectories in data directory
        classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
        print(f"Found classes: {classes}")
        print(f"Number of classes: {len(classes)}")

        # Count samples per class
        class_counts = {}
        for class_name in classes:
            class_path = os.path.join(DATA_DIR, class_name)
            images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.jpeg', '.png', '.gif'))]
            class_counts[class_name] = len(images)
            print(f"  {class_name}: {len(images)} images")

        total_samples = sum(class_counts.values())
        print(f"\nTotal samples: {total_samples} images")

        if total_samples == 0:
            print("Error: No training samples found")
            exit(1)

        # Data preprocessing
        transform_train = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        transform_val = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Load dataset
        print("\nLoading dataset...")
        dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform_train)
        print(f"Dataset size: {len(dataset)}")

        # Split into train and validation sets
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
        val_dataset.dataset.transform = transform_val

        # Create data loaders
        batch_size = 32
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

        print(f"Train set size: {len(train_dataset)} images")
        print(f"Validation set size: {len(val_dataset)} images")
        print(f"Batch size: {batch_size}")
        print(f"Number of train batches: {len(train_loader)}")
        print(f"Number of validation batches: {len(val_loader)}")

        # Build model
        print("\nBuilding model...")
        model = models.resnet18(pretrained=True)
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, len(classes))
        model.fc.requires_grad = True

        # Device configuration
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)
        print(f"Using device: {device}")

        # Loss function and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), 
                              lr=0.001, momentum=0.9, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

        # Training parameters
        epochs = 10

        print(f"\nStarting training, {epochs} epochs total...")

        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            
            # Training phase
            model.train()
            running_loss = 0.0
            train_correct = 0
            train_total = 0
            
            for batch_idx, (inputs, labels) in enumerate(train_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
                
                if batch_idx % 10 == 0:
                    print(f"  Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")
            
            train_loss = running_loss / len(train_dataset)
            train_accuracy = 100 * train_correct / train_total
            
            # Validation phase
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * inputs.size(0)
                    
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            
            val_loss = val_loss / len(val_dataset)
            val_accuracy = 100 * correct / total
            
            scheduler.step()
            
            print(f"  Training loss: {train_loss:.4f}, Training accuracy: {train_accuracy:.2f}%")
            print(f"  Validation loss: {val_loss:.4f}, Validation accuracy: {val_accuracy:.2f}%")

        print("\nTraining completed!")
        print(f"Final validation accuracy: {val_accuracy:.2f}%")

        # Save model
        model_save_path = 'baseline_resnet18_cleaned.pth'
        print(f"\nSaving model to: {model_save_path}")
        print(f"Current working directory: {os.getcwd()}")
        
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_accuracy,
            'classes': classes
        }, model_save_path)

        print(f"Model saved successfully!")
        print(f"Model file size: {os.path.getsize(model_save_path) / 1024 / 1024:.2f} MB")

    except Exception as e:
        print(f"\nError during training:")
        print(traceback.format_exc())


=== Using cleaned_images to train ResNet18 model ===
Current working directory: D:\jupyter
Data directory: dataset_splits/cleaned_images
Data directory exists: True
Found classes: ['biotite', 'bornite', 'chrysocolla', 'malachite', 'muscovite', 'pyrite', 'quartz']
Number of classes: 7
  biotite: 41 images
  bornite: 100 images
  chrysocolla: 92 images
  malachite: 144 images
  muscovite: 56 images
  pyrite: 66 images
  quartz: 82 images

Total samples: 581 images

Loading dataset...
Dataset size: 580
Train set size: 464 images
Validation set size: 116 images
Batch size: 32
Number of train batches: 15
Number of validation batches: 4

Building model...


C:\Users\1\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\1\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using device: cuda

Starting training, 10 epochs total...

Epoch 1/10
  Batch 0/15, Loss: 2.2141
  Batch 10/15, Loss: 2.1758
  Training loss: 2.0030, Training accuracy: 21.98%
  Validation loss: 1.8092, Validation accuracy: 24.14%

Epoch 2/10
  Batch 0/15, Loss: 1.3402
  Batch 10/15, Loss: 1.9013
  Training loss: 1.5139, Training accuracy: 48.49%
  Validation loss: 1.4545, Validation accuracy: 46.55%

Epoch 3/10
  Batch 0/15, Loss: 1.4073
  Batch 10/15, Loss: 1.0539
  Training loss: 1.2526, Training accuracy: 61.21%
  Validation loss: 1.2975, Validation accuracy: 50.00%

Epoch 4/10
  Batch 0/15, Loss: 0.9878
  Batch 10/15, Loss: 1.0503
  Training loss: 1.0494, Training accuracy: 71.12%
  Validation loss: 1.1438, Validation accuracy: 59.48%

Epoch 5/10
  Batch 0/15, Loss: 0.8063
  Batch 10/15, Loss: 0.8433
  Training loss: 0.9136, Training accuracy: 74.57%
  Validation loss: 1.0488, Validation accuracy: 67.24%

Epoch 6/10
  Batch 0/15, Loss: 0.8572
  Batch 10/15, Loss: 0.6795
  Training